In [1]:
import pandas as pd
import pickle

import ipywidgets as widgets
from ipywidgets import interact

from src.data.data_load import (
    load_tables, 
    load_online_instance, 
    load_distances, 
    upload_ONLINE_static_solution
)
from src.data.solution_load import load_solution_dfs, _include_all_city
from src.utils.filtering import flexible_filter
from src.utils.plotting import plot_metrics_comparison_dynamic
from src.data.metrics import collect_results_to_df, compute_metrics_with_moves, get_day_plotting_df
from src.config.experimentation_config import *
from src.config.SD_experimentation_config import *
from src.config.config import *

data_path = '../data'

distance_type = 'osrm'              # Options: ['osrm', 'manhattan']
dist_method = 'haversine'      # Options: ['precalced', 'haversine']

optimization_obj = 'driver_distance'

directorio_df, labors_raw_df, cities_df, duraciones_df, valid_cities = load_tables(data_path, generate_labors=False)
# dist_dict = load_distances(data_path, distance_type, instance, dist_method)

metricas = ['service_count', 'vt_count', 'num_drivers', 'driver_extra_time', 'driver_move_distance']


# Upload results

In [2]:
def upload_simulated_instances():
    
    instances = {}

    for n_serv in n_services:
        labors_real_dfs = pd.DataFrame()
        labors_static_dfs = pd.DataFrame()
        labors_dynamic_dfs = pd.DataFrame()
        for scenario in scenarios:
            for seed in seeds:
                instance = f'N{n_serv}/{scenario}/seed_{seed}'
                labors_real_df, labors_static_df, labors_dynamic_df = load_online_instance(data_path, instance, labors_raw_df)

                for df in [labors_real_df, labors_static_df, labors_dynamic_df]:
                    df['n_serv'] = n_serv
                    df['scenario'] = scenario
                    df['seed'] = seed
                
                labors_real_df = _include_all_city(labors_real_df)
                labors_static_df = _include_all_city(labors_static_df)
                labors_dynamic_df = _include_all_city(labors_dynamic_df)

                labors_real_dfs = pd.concat([labors_real_dfs, labors_real_df])
                labors_static_dfs = pd.concat([labors_static_dfs, labors_static_df])
                labors_dynamic_dfs = pd.concat([labors_dynamic_dfs, labors_dynamic_df])
        
        instances[n_serv] = (labors_real_dfs, labors_static_dfs, labors_dynamic_dfs)

    return instances
        
instances = upload_simulated_instances()    


# Instance exploration

## Global

In [7]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, Dropdown


def plot_topology_reactive(instances, n_services_list, scenarios, seeds):
    """
    Creates two reactive dropdowns: n_services and city.
    Whenever the user selects a new value, the plots update.
    """

    # ---- helper: available cities for a given n_services ----
    def available_cities(n):
        df_real = instances[n][0]
        if "city" in df_real.columns:
            return sorted(df_real["city"].dropna().unique())
        return []

    # ---- reactive plotting function ----
    def draw(n_services, city):
        df_real, df_static, df_dynamic = instances[n_services]

        # ---------------- FILTER BY CITY ----------------
        dfR = df_real[df_real['city'] == city].copy()
        dfS = df_static[df_static['city'] == city].copy()
        dfD = df_dynamic[df_dynamic['city'] == city].copy()

        # ---------------- HISTOGRAM DATA ----------------
        hist_rows = []
        for sc in scenarios:
            sub = dfR[dfR["scenario"] == sc]
            for seed, g in sub.groupby("seed"):
                vt_count = int((g["labor_category"] == "VEHICLE_TRANSPORTATION").sum())
                hist_rows.append({"scenario": sc, "seed": seed, "vt": vt_count})

        hist_df = pd.DataFrame(hist_rows)

        # ---------------- BOX PLOT DATA ----------------
        box_rows = []
        for sc in scenarios:
            df_stat_sc = dfS[dfS["scenario"] == sc]
            df_dyn_sc = dfD[dfD["scenario"] == sc]

            stat_by_seed = df_stat_sc.groupby("seed")["service_id"].nunique()
            dyn_by_seed = df_dyn_sc.groupby("seed")["service_id"].nunique()

            union_seeds = sorted(set(stat_by_seed.index) | set(dyn_by_seed.index))
            for seed in union_seeds:
                n_static = int(stat_by_seed.get(seed, 0))
                n_dynamic = int(dyn_by_seed.get(seed, 0))
                total = n_static + n_dynamic
                prop_static = (n_static / total) if total > 0 else np.nan
                box_rows.append({
                    "scenario": sc,
                    "seed": seed,
                    "prop_static": prop_static
                })

        box_df = pd.DataFrame(box_rows)

        # ---------------- PLOTTING ----------------
        fig = make_subplots(
            rows=2, cols=3,
            subplot_titles=[f"{sc}" for sc in scenarios] +
                           ["Proporción servicios offline"],
            specs=[[{"type": "xy"}]*3,
                   [{"colspan": 3, "type": "xy"}, None, None]],
            vertical_spacing=0.15
        )

        # -- top row histograms --
        bins = 20
        for i, sc in enumerate(scenarios):
            h = hist_df[hist_df.scenario == sc]
            if not h.empty:
                vals, edges = np.histogram(h["vt"].values, bins=bins)
                centers = (edges[:-1] + edges[1:]) / 2
                fig.add_trace(
                    go.Bar(x=centers, y=vals, showlegend=False),
                    row=1, col=i+1
                )
            fig.update_xaxes(title_text="Labores de transp.", row=1, col=i+1)
            fig.update_yaxes(title_text="Frecuencia", row=1, col=i+1)

        # -- bottom row boxplot --
        for sc in scenarios:
            fig.add_trace(
                go.Box(
                    y=box_df[box_df.scenario == sc]["prop_static"],
                    name=sc,
                    boxmean="sd"
                ),
                row=2, col=1
            )

        fig.update_layout(
            height=700,
            width=1200,
            title=dict(
                text="Caracterización instancias simuladas (120 servicios)",
                x=0.5,              # ← center horizontally
                xanchor="center"    # ← treat x as the center
            ),
            showlegend=False,
            template="plotly_white"
        )


        fig.show()

    # ---- WIDGETS ----
    # Default values
    default_n = n_services_list[0]
    default_city = available_cities(default_n)[0]

    # Use interact to automatically re-run draw() on change
    interact(
        draw,
        n_services=Dropdown(options=n_services_list, value=default_n, description="N services"),
        city=Dropdown(options=available_cities(default_n), value='ALL', description="City")
    )

plot_topology_reactive(
    instances=instances,
    n_services_list=n_services,
    scenarios=scenarios,
    seeds=seeds
)


interactive(children=(Dropdown(description='N services', options=(120, 160, 200, 240, 280, 320), value=120), D…

# Result visualization

In [8]:
algo_selection = {
    'OFFLINE': False,
    'INSERT': True,
    'BUFFER_FIXED': True,
    'REACT': False,
    'BUFFER_REACT': True,
}

## Load results

In [ ]:
import pickle
import pandas as pd

import os

algorithms = []

for algo, flag in algo_selection.items():
    if flag:
        algorithms.append({
            "name": algo,
            "type": "algorithm",
            "color": algo_colors[algo],
            "visible": True,
        })

def load_all_results_explicit(
    n_services_list,
    scenarios_list,
    seeds_list,
    dist_methods_list,
    algorithms
) -> pd.DataFrame:
    """
    Clean, explicit loader for result files located in:
    root/N{n_services}/{scenario}/seed_{seed}/{dist_method}/{algorithm}.pkl

    Returns a tidy DataFrame with columns:
    [n_services, scenario, seed, dist_method, algorithm, labors_df, moves_df, metadata]
    """
    root = f'{REPO_PATH}/data/resultados/simulation'
    rows = []

    for n_services in n_services_list:
        n_dir = os.path.join(root, f'N{n_services}')

        for scenario in scenarios_list:
            
            scenario_dir = os.path.join(n_dir, scenario)

            for seed in seeds_list:
                seed_dir = os.path.join(scenario_dir, f"seed_{seed}")

                for dist_method in dist_methods_list:
                    dist_dir = os.path.join(seed_dir, dist_method)

                    for algorithm in algorithms:
                        pkl_path = os.path.join(dist_dir, f"res_algo_{algorithm}.pkl")

                        # Load pickle
                        try:
                            with open(pkl_path, "rb") as f:
                                labors_df, moves_df, metadata = pickle.load(f)
                        except Exception as e:
                            print(f"❌ Error loading {pkl_path}: {e}")
                            continue

                        labors_df = _include_all_city(labors_df)
                        moves_df = _include_all_city(moves_df)

                        rows.append({
                            "n_services": n_services,
                            "scenario": scenario,
                            "seed": seed,
                            "dist_method": dist_method,
                            "algorithm": algorithm,
                            "labors_df": labors_df,
                            "moves_df": moves_df,
                            "metadata": metadata,
                            "path": str(pkl_path)
                        })

    
    return pd.DataFrame(rows)

runs_df = load_all_results_explicit(
    n_services_list=n_services+[900],
    scenarios_list=scenarios,
    seeds_list=seeds,
    dist_methods_list=['haversine'],
    algorithms=[algo for algo, flag in algo_selection.items() if flag]
)


## Global results

## Box plots

In [16]:
# plot_results_boxplots.py
import os
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from ipywidgets import interact, widgets

# --- you must have this function available in scope ---
# from src.metrics import compute_metrics_with_moves
# (the user already has compute_metrics_with_moves in their codebase)

def build_metrics_df_for_plots(
    runs_df: pd.DataFrame,
    n_services: int,
    city: str,
    metricas: List[str],
    dist_dict: Dict,
    workday_hours: int = 8,
    agg_fn = np.sum
) -> pd.DataFrame:
    """
    Build a tidy dataframe of metric values ready for plotting.

    Parameters
    ----------
    runs_df : pd.DataFrame
        Tidy table with the following required columns:
          - 'n_services' (int)
          - 'scenario' (str)
          - 'seed' (int)
          - 'algorithm' (str)
          - 'labors_df' (pd.DataFrame)
          - 'moves_df' (pd.DataFrame)
        (If your runs_df stores labors/moves differently, you must first
         transform it into this form.)
    n_services : int
        Which experiment size to plot.
    city : str
        City code or 'ALL' (passed to compute_metrics_with_moves).
    metricas : list[str]
        List of metric names (same as used in compute_metrics_with_moves).
    dist_dict : dict
        Distances dictionary required by compute_metrics_with_moves.
    workday_hours : int
        Workday hours parameter forwarded to metric computation.
    agg_fn : callable
        Aggregation function to reduce daily series into a single scalar per run (default: sum).

    Returns
    -------
    pd.DataFrame
        tidy df with columns:
        ['n_services','scenario','seed','algorithm','metric','value','city']
    """
    required_cols = {"n_services", "scenario", "seed", "algorithm", "labors_df", "moves_df"}
    if not required_cols.issubset(set(runs_df.columns)):
        raise ValueError(f"runs_df must contain columns: {required_cols}")

    # filter by n_services
    subset = runs_df.loc[runs_df["n_services"] == n_services].copy()
    if subset.empty:
        raise ValueError(f"No runs found for n_services={n_services}")

    rows = []
    # iterate runs
    for idx, run in subset.iterrows():
        algo = run["algorithm"]
        scn = run["scenario"]
        seed = run["seed"]
        n_s = run["n_services"]
        labors_df = run["labors_df"]
        moves_df = run["moves_df"]

        # compute metrics using your function (it returns a dataframe indexed by day)
        metrics_df = compute_metrics_with_moves(
            labors_df,
            moves_df,
            fechas=['2026-11-11'],            # None -> function should compute for available dates (if your func requires fechas, adapt)
            dist_dict=dist_dict,
            workday_hours=workday_hours,
            city=city,
            skip_weekends=False,
            assignment_type='algorithm',
            dist_method='haversine'
        )

        # If metrics_df is empty, skip
        if metrics_df is None or metrics_df.empty:
            # produce NaNs for requested metrics
            for m in metricas:
                rows.append({
                    "n_services": n_s, "scenario": scn, "seed": seed, "algorithm": algo,
                    "metric": m, "value": np.nan, "city": city
                })
            continue

        # For each metric, reduce to scalar using agg_fn
        for m in metricas:
            if m not in metrics_df.columns:
                val = np.nan
            else:
                series = pd.to_numeric(metrics_df[m], errors='coerce').dropna()
                val = agg_fn(series) if not series.empty else 0.0
            rows.append({
                "n_services": n_s, "scenario": scn, "seed": seed, "algorithm": algo,
                "metric": m, "value": val, "city": city
            })

    tidy = pd.DataFrame(rows)
    # ensure scenario ordering (if you want a particular order: easy, normal, hard)
    scenario_order = ["easy", "normal", "hard"]
    present = [s for s in scenario_order if s in tidy['scenario'].unique()]
    tidy['scenario'] = pd.Categorical(tidy['scenario'], categories=present + [c for c in tidy['scenario'].unique() if c not in present], ordered=True)
    return tidy


def plot_simulation_metrics_boxplots(
    metrics_tidy_df: pd.DataFrame,
    metricas: List[str],
    algorithms_info: List[Dict[str, str]],
    scenarios: List[str],
    save: bool = False,
    save_dir: Optional[str] = None,
    title_prefix: Optional[str] = None,
):
    """
    Instead of one multi-row figure, this function produces
    ONE FIGURE PER METRIC.

    Each figure shows:
    - x-axis: scenario
    - grouped boxplots: one box per algorithm per scenario
    - shared colors across all figures
    """
    # alg_colors stays the same
    alg_colors = {a['name']: a.get('color', None) for a in algorithms_info}

    # NEW: preserve the exact algorithm order provided
    alg_order = [a['name'] for a in algorithms_info]
    algs_present = [a for a in alg_order if a in metrics_tidy_df['algorithm'].unique()]


    for metric in metricas:
        dfm = metrics_tidy_df[metrics_tidy_df['metric'] == metric].copy()

        # Create an empty figure
        fig = go.Figure()

        # Order scenarios if available
        scenario_order = scenarios if all(s in dfm['scenario'].unique() for s in scenarios) \
                                    else sorted(dfm['scenario'].unique())

        # One trace per algorithm
        for alg in algs_present:
            alg_df = dfm[dfm['algorithm'] == alg]
            if alg_df.empty:
                continue

            fig.add_trace(
                go.Box(
                    x=alg_df['scenario'].astype(str),
                    y=alg_df['value'],
                    name=alg,
                    marker_color=alg_colors.get(alg),
                    boxmean="sd",
                    legendgroup=alg,
                    hovertemplate="scenario=%{x}<br>"
                                  "algorithm=%{fullData.name}<br>"
                                  "value=%{y:.2f}<extra></extra>",
                )
            )

        # Layout
        titles = {
            'service_count': 'Número de servicios',
            'vt_count': 'Número labores Vehicle Transportation',
            'num_drivers': 'Número conductores',
            'driver_extra_time': 'Tiempo extra', 
            'driver_move_distance': 'Distancia en vacío'
        }

        full_title = titles[metric]
        fig.update_layout(
            title=dict(
                text=full_title,
                x=0.5,              # ← center horizontally
                xanchor="center"    # ← treat x as the center
            ),
            yaxis_title=metric,
            boxmode="group",
            width=1100,
            height=450,
            plot_bgcolor="white",
            legend=dict(
                orientation="h",
                yanchor="bottom", y=1.05,
                xanchor="center", x=0.5
            ),
            margin=dict(t=100, b=60),
        )

        fig.update_yaxes(
            showgrid=True,
            gridcolor="rgba(0,0,0,0.25)",   # very subtle light gray
            gridwidth=0.5,                 # thin lines
            griddash="dash",
            zeroline=False                 # remove bold zero-line
        )


        fig.update_xaxes(
            categoryorder="array",
            categoryarray=scenario_order,
            tickangle=-15
        )

        # Save if requested
        if save and save_dir:
            os.makedirs(save_dir, exist_ok=True)
            fname = f"{metric}_boxplot.html"
            out_path = os.path.join(save_dir, fname)
            fig.write_html(out_path, include_plotlyjs="cdn")
            print(f"Saved plot to {out_path}")
        else:
            fig.show()

# ---------------------------
# Interactive wrapper
# ---------------------------
def interactive_metrics_boxplots(
    runs_df: pd.DataFrame,
    n_services_list: List[int],
    scenarios: List[str],
    algorithms_info: List[Dict[str, str]],
    metricas: List[str],
    dist_dict: Dict,
    city_options: List[str] = None,
):
    """
    Build ipywidgets.interact UI to choose n_services and city.
    runs_df must be tidy as described in build_metrics_df_for_plots docstring.
    """

    if city_options is None:
        # infer from runs_df (there may be many cities inside labors_df; present a union)
        city_options = ["ALL"]
        # try to infer cities present in run labors_df objects
        cities_set = set()
        for _, r in runs_df.iterrows():
            try:
                ldf = r['labors_df']
                if 'city' in ldf.columns:
                    cities_set.update(ldf['city'].unique().tolist())
            except Exception:
                continue
        city_options += sorted(list(cities_set))

    n_services_dropdown = widgets.Dropdown(
        options=sorted(n_services_list),
        value=sorted(n_services_list)[0],
        description='n_services:',
        style={"description_width": "initial"}
    )

    city_dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='City:',
        style={"description_width": "initial"}
    )

    def _update(n_services, city):
        # build metrics table
        tidy = build_metrics_df_for_plots(
            runs_df=runs_df,
            n_services=n_services,
            city=city,
            metricas=metricas,
            dist_dict=dist_dict
        )
        # for title: include counts
        title = f"city={city}"
        plot_simulation_metrics_boxplots(
            metrics_tidy_df=tidy,
            metricas=metricas,
            algorithms_info=algorithms_info,
            scenarios=scenarios,
            title_prefix=title
        )

    interact(_update, n_services=n_services_dropdown, city=city_dropdown)


In [ ]:
interactive_metrics_boxplots(
    runs_df,
    n_services+[900],
    scenarios,
    algorithms,
    metricas,
    dict(),
    ['ALL'] + valid_cities,
)

interactive(children=(Dropdown(description='n_services:', options=(120, 160, 200, 240, 280, 320, 900), style=D…

## EVALUATING

In [10]:
labors_hist_df, moves_hist_df, postponed_labors_hist = load_solution_dfs(
    data_path, 
    'instAD3', 
    dist_method,
    algorithm='Histórico',
    include_all_city=True
)
reference_df = (
    moves_hist_df
    .query("labor_category == 'DRIVER_MOVE'")
    .drop_duplicates(["service_id", "city"])
    .groupby("city")
    .agg(
        n_services=("service_id", "count"),
        driver_move_distance=("distance_km", "sum")
    )
    .div(7)
    .reset_index()
)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown

def get_reference_values(reference_df, city):
    """Return (ref_n, ref_dist) for the selected city, or (None, None) if not found."""
    row = reference_df[reference_df["city"] == city]
    if row.empty:
        return None, None
    r = row.iloc[0]
    return r["n_services"], r["driver_move_distance"]


def build_metric_driver_distance_means(
    runs_df: pd.DataFrame,
    scenario: str,
    city: str,
    dist_dict: dict,
    algorithms_info: list,
    workday_hours: int = 8
):
    """
    Returns tidy df:
    ['algorithm','n_services','mean_driver_move_distance']
    filtered by scenario AND city.
    """

    # Filter by scenario and city
    df = runs_df[(runs_df["scenario"] == scenario)].copy()
    if df.empty:
        raise ValueError(f"No runs found for scenario='{scenario}', city='{city}'")

    rows = []

    for idx, r in df.iterrows():
        algo = r["algorithm"]
        n_s = r["n_services"]

        filtered_labors = r["labors_df"][r["labors_df"]['city'] == city]
        filtered_moves = r["moves_df"][r["moves_df"]['city'] == city]

        metrics_df = compute_metrics_with_moves(
            r["labors_df"],
            r["moves_df"],
            fechas=['2026-11-11'],
            dist_dict=dist_dict,
            workday_hours=workday_hours,
            city=city,   # ← NOW WE USE THE CITY HERE
            skip_weekends=False,
            assignment_type="algorithm",
            dist_method="haversine"
        )

        if metrics_df is None or metrics_df.empty:
            val = np.nan
        else:
            series = pd.to_numeric(metrics_df["driver_move_distance"], errors='coerce').dropna()
            val = series.sum() if not series.empty else 0.0

        rows.append({
            "algorithm": algo,
            "n_services": n_s,
            "value": val
        })

    tidy = pd.DataFrame(rows)
    mean_df = tidy.groupby(["algorithm","n_services"], as_index=False)["value"].mean()
    mean_df.rename(columns={"value": "mean_driver_move_distance"}, inplace=True)

    return mean_df


def plot_driver_distance_means(
    mean_df: pd.DataFrame,
    algorithms_info: list,
    scenario: str,
    city: str,
    reference_df: pd.DataFrame   # ← added
):
    """Plot mean driver move distance vs n_services for a scenario + city."""

    alg_colors = {a["name"]: a.get("color", None) for a in algorithms_info}

    def dash_style_for(algo_dict):
        name = algo_dict["name"].lower()
        t = algo_dict.get("type","")
        if t == "historic":
            return "solid"
        elif "react" in name:
            return "dashdot"
        else:
            return "dash"

    fig = go.Figure()
    alg_order = [a["name"] for a in algorithms_info]

    # --------------------------------------------------------
    # PLOT LINES FOR ALL ALGORITHMS (unchanged)
    # --------------------------------------------------------
    for alg in alg_order:
        algo_info = next(a for a in algorithms_info if a["name"] == alg)
        style = dash_style_for(algo_info)

        sub = mean_df[mean_df["algorithm"] == alg]
        if sub.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=sub["n_services"],
                y=sub["mean_driver_move_distance"],
                mode="lines+markers",
                name=alg,
                marker=dict(size=8),
                line=dict(
                    width=3,
                    dash=style,
                    color=alg_colors.get(alg)
                )
            )
        )

    # --------------------------------------------------------
    # ADD REFERENCE LINES (the new part)
    # --------------------------------------------------------
    ref_n, ref_dist = get_reference_values(reference_df, city)

    if ref_n is not None:
        # Vertical reference line
        fig.add_vline(
            x=ref_n,
            line_width=2,
            line_dash="dot",
            line_color="#C0392B",
            annotation_text=f"Ref n={ref_n:.1f}",
            annotation_position="top left"
        )

    if ref_dist is not None:
        # Horizontal reference line
        fig.add_hline(
            y=ref_dist,
            line_width=2,
            line_dash="dot",
            line_color="#C0392B",
            annotation_text=f"Ref dist={ref_dist:.1f}",
            annotation_position="bottom right"
        )

    # --------------------------------------------------------
    # Layout (unchanged)
    # --------------------------------------------------------
    fig.update_layout(
        title=f"Distancia en vacío promedio",
        xaxis_title="Tamaño de instancia (n_services)",
        yaxis_title="Distancia en vacío (km)",
        plot_bgcolor="white",
        width=1150,
        height=550,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.1,
            xanchor="center",
            x=0.5,
            font=dict(size=11)
        ),
        margin=dict(t=90, b=60)
    )

    fig.update_xaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.25)",
    gridwidth=1.2,
    griddash="dash",
    zeroline=False
)

    fig.update_yaxes(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.25)",
        gridwidth=1.2,
        griddash="dash",
        zeroline=False
    )

    fig.show()


def interactive_driver_distance_plot(
    runs_df: pd.DataFrame,
    scenarios: list,
    cities: list,
    algorithms_info: list,
    dist_dict: dict,
    reference_df: pd.DataFrame      # ← add this
):
    scenario_dropdown = Dropdown(
        options=scenarios,
        value=scenarios[0],
        description="Scenario:"
    )

    city_dropdown = Dropdown(
        options=cities,
        value=cities[0],
        description="City:"
    )

    def _update(selected_scenario, selected_city):
        mean_df = build_metric_driver_distance_means(
            runs_df=runs_df,
            scenario=selected_scenario,
            city=selected_city,
            dist_dict=dist_dict,
            algorithms_info=algorithms_info
        )

        # mean_df.to_csv(f'./distancia_vs_servicios_scn_{selected_scenario}.csv')

        plot_driver_distance_means(
            mean_df=mean_df,
            algorithms_info=algorithms_info,
            scenario=selected_scenario,
            city=selected_city,
            reference_df=reference_df
        )

    interact(
        _update,
        selected_scenario=scenario_dropdown,
        selected_city=city_dropdown
    )




interactive_driver_distance_plot(
    runs_df,
    scenarios,
    ['ALL']+valid_cities,
    algorithms,
    {},
    reference_df)

interactive(children=(Dropdown(description='Scenario:', options=('easy', 'normal', 'hard'), value='easy'), Dro…

In [21]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown

def get_reference_values(reference_df, city):
    """Return (ref_n, ref_dist) for the selected city, or (None, None) if not found."""
    row = reference_df[reference_df["city"] == city]
    if row.empty:
        return None, None
    r = row.iloc[0]
    return r["n_services"], r["driver_move_distance"]


def build_metric_driver_distance_means(
    runs_df: pd.DataFrame,
    scenario: str,
    city: str,
    dist_dict: dict,
    algorithms_info: list,
    workday_hours: int = 8
):
    """
    Returns df with:
    ['algorithm','n_services','mean','std','N','ci_low','ci_high']
    """

    df = runs_df[
        (runs_df["scenario"] == scenario)
    ].copy()

    if df.empty:
        raise ValueError(f"No runs found for scenario='{scenario}'")

    rows = []

    for _, r in df.iterrows():
        algo = r["algorithm"]
        n_s = r["n_services"]

        labors_df = r["labors_df"]
        moves_df = r["moves_df"]

        # Compute metric
        metrics_df = compute_metrics_with_moves(
            labors_df,
            moves_df,
            fechas=['2026-11-11'],
            dist_dict=dist_dict,
            workday_hours=workday_hours,
            city=city,
            skip_weekends=False,
            assignment_type="algorithm",
            dist_method="haversine"
        )

        if metrics_df is None or metrics_df.empty:
            val = np.nan
        else:
            series = pd.to_numeric(metrics_df["driver_move_distance"], errors='coerce').dropna()
            val = series.sum() if not series.empty else 0.0

        rows.append({
            "algorithm": algo,
            "n_services": n_s,
            "value": val
        })

    tidy = pd.DataFrame(rows)

    # Summary stats
    agg = tidy.groupby(["algorithm","n_services"])["value"].agg(['mean','std','count']).reset_index()
    agg.rename(columns={"count":"N"}, inplace=True)

    # Compute CI bounds
    z = 1.96
    agg["ci_low"] = agg["mean"] - z * (agg["std"] / np.sqrt(agg["N"]))
    agg["ci_high"] = agg["mean"] + z * (agg["std"] / np.sqrt(agg["N"]))

    return agg


def plot_driver_distance_means(
    mean_df: pd.DataFrame,
    algorithms_info: list,
    scenario: str,
    city: str,
    reference_df: pd.DataFrame   # columns: ["city","n_services","driver_move_distance"]
):
    """
    Plots mean driver distance with dashed lines + whisker std dev on markers
    and adds vertical/horizontal reference lines for the chosen city.
    """

    # -------------------------------
    #   Fetch reference city values
    # -------------------------------
    ref = reference_df[reference_df["city"] == city]
    ref_n = None
    ref_val = None

    if not ref.empty:
        ref_n = float(ref.iloc[0]["n_services"])
        ref_val = float(ref.iloc[0]["driver_move_distance"])

    # Colors by algorithm
    alg_colors = {a["name"]: a.get("color") for a in algorithms_info}

    # Determine dashed style
    def dash_style_for(a):
        name = a["name"].lower()
        t = a.get("type", "")
        if t == "historic":
            return "solid"
        elif "react" in name:
            return "dashdot"
        else:
            return "dash"

    fig = go.Figure()

    # -------------------------------
    #   Plot algorithms
    # -------------------------------
    for algo in algorithms_info:
        name = algo["name"]
        color = alg_colors[name]
        style = dash_style_for(algo)

        sub = mean_df[mean_df["algorithm"] == name]
        if sub.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=sub["n_services"],
                y=sub["mean"],
                mode="lines+markers",
                name=name,
                line=dict(color=color, dash=style, width=3),
                marker=dict(size=8),

                # --- Whiskers (vertical std) ---
                error_y=dict(
                    type="data",
                    array=sub["std"],         # +std
                    arrayminus=sub["std"],    # -std
                    visible=True,
                    thickness=1.4,
                    width=8,   # whisker cap width
                    color=color,
                    # opacity=0.55,
                ),
            )
        )

    # -------------------------------
    #   Reference lines (if city exists)
    # -------------------------------
    if ref_n is not None:
        fig.add_vline(
            x=ref_n,
            line_width=2,
            line_dash="dot",
            line_color="#C0392B",
            annotation_text=f"n={int(ref_n)}",
            annotation_position="top"
        )

    if ref_val is not None:
        fig.add_hline(
            y=ref_val,
            line_width=2,
            line_dash="dot",
            line_color="#C0392B",
            annotation_text=f"{ref_val:.0f}",
            annotation_position="right"
        )

    # -------------------------------
    #   Layout
    # -------------------------------
    fig.update_layout(
        title=f"Mean Driver Move Distance",
        xaxis_title="Instance Size (n_services)",
        yaxis_title="Mean Driver Move Distance",
        plot_bgcolor="white",
        width=1150,
        height=550,
        legend=dict(
            orientation="h",
            y=1.12,
            x=0.5,
            xanchor="center",
            font=dict(size=11)
        ),
        margin=dict(t=80, b=60),
    )

    fig.update_xaxes(showgrid=True, gridcolor="#E0E0E0")
    fig.update_yaxes(showgrid=True, gridcolor="#E0E0E0")

    fig.show()





def interactive_driver_distance_plot(
    runs_df: pd.DataFrame,
    scenarios: list,
    cities: list,
    algorithms_info: list,
    dist_dict: dict,
    reference_df: pd.DataFrame      # ← add this
):
    scenario_dropdown = Dropdown(
        options=scenarios,
        value=scenarios[0],
        description="Scenario:"
    )

    city_dropdown = Dropdown(
        options=cities,
        value=cities[0],
        description="City:"
    )

    def _update(selected_scenario, selected_city):
        mean_df = build_metric_driver_distance_means(
            runs_df=runs_df,
            scenario=selected_scenario,
            city=selected_city,
            dist_dict=dist_dict,
            algorithms_info=algorithms_info
        )

        plot_driver_distance_means(
            mean_df=mean_df,
            algorithms_info=algorithms_info,
            scenario=selected_scenario,
            city=selected_city,
            reference_df=reference_df
        )

    interact(
        _update,
        selected_scenario=scenario_dropdown,
        selected_city=city_dropdown
    )


interactive_driver_distance_plot(
    runs_df,
    scenarios,
    ['ALL']+valid_cities,
    algorithms,
    {},
    reference_df)

interactive(children=(Dropdown(description='Scenario:', options=('easy', 'normal', 'hard'), value='easy'), Dro…